# Agente Extrator de Metadados Bibliográficos

Pipeline com **CrewAI** que localiza um PDF em uma pasta, extrai seu texto e gera uma entrada bibliográfica estruturada (JSON) usando LLM.

## 1. Configuração inicial
Suprime avisos para manter a saída limpa.

In [4]:
import warnings

warnings.filterwarnings('ignore')

## 2. Variáveis de ambiente
Carrega a `NVIDIA_API_KEY` do `.env` e a expõe como `NVIDIA_NIM_API_KEY` para o SDK da NVIDIA.

In [5]:
import os
from dotenv import load_dotenv


load_dotenv()

api_key = os.getenv("NVIDIA_API_KEY")

os.environ["NVIDIA_NIM_API_KEY"] = api_key

## 3. Inicialização do LLM
Configura o modelo **Llama 3.1 70B Instruct** via NVIDIA NIM com temperatura baixa (0.2) para respostas mais deterministicas.

In [6]:
import os
from crewai import LLM, Agent


llm = LLM(
    model="nvidia_nim/meta/llama-3.1-70b-instruct",
    temperature=0.2
)


## 4. Ferramenta de leitura de PDF e de salvar arquivo .Bib
Tool `read_pdf_content` customizada com `pypdf` que extrai texto de todas as páginas de um PDF. Retorna erro se o arquivo for vazio ou contiver apenas imagens.

Tool `save_bib_file` 

## 5. Schema bibliográfico (Pydantic)
Define `BibliographyEntry` com campos como `title`, `authors`, `year`, `doi`, etc. Usado como `output_json` para garantir que a LLM retorne dados estruturados.

In [7]:
from typing import Optional, List
from pydantic import BaseModel, Field


class BibliographyEntry(BaseModel):
    """Json schema for a single bibliography entry."""
    
    entry_type: str = Field(description="The type of publication (e.g., article, inproceedings, book, dataset).")
    citation_key: str = Field(description="A unique citation key, usually formatted as 'PrimaryAuthorYear' (e.g., Smith2024).")
    title: str = Field(description="The exact title of the academic paper.")
    authors: List[str] = Field(description="List of all authors in the format 'Lastname, Firstname'.")
    journal_or_booktitle: Optional[str] = Field(default=None, description="The journal name, conference proceedings, or book title where it was published.")
    year: int = Field(description="The year of publication.")
    volume: Optional[str] = Field(default=None, description="The volume number.")
    number: Optional[str] = Field(default=None, description="The issue number.")
    pages: Optional[str] = Field(default=None, description="The page range (e.g., 100-115).")
    publisher: Optional[str] = Field(default=None, description="The publisher or organization.")
    doi: Optional[str] = Field(default=None, description="The DOI (Digital Object Identifier) if available.")

In [ ]:
import os
from typing import List, Optional
from crewai.tools import BaseTool
from pydantic import BaseModel
from crewai.tools import (tool, BaseTool)
import pypdf

@tool("PDF Reader Tool")
def read_pdf_content(file_path: str) -> str:
    """Reads and extracts all the text content from a PDF file. 
    Use this tool when you need to read the content of a .pdf file."""
    try:
        reader = pypdf.PdfReader(file_path)
        extracted_text = ""
        
        for page_num, page in enumerate(reader.pages):
            page_text = page.extract_text()
            if page_text:
                extracted_text += f"\n--- Page {page_num + 1} ---\n" + page_text
                
        if not extracted_text.strip():
            return "Error: The PDF file seems to be empty or contains only scanned images (no selectable text)."
            
        return extracted_text
        
    except Exception as e:
        return f"Error reading PDF file: {str(e)}"
    




## 6. Definição dos agentes
- **Team Archivist**: lista arquivos na pasta `./data`, encontra o melhor match (inclusive com typos) e lê o conteúdo.
- **Academic Metadata Extractor**: recebe o texto bruto e extrai metadados bibliográficos.

In [9]:
from crewai_tools import DirectoryReadTool

from crewai import Agent, Task, Crew, Process

directory_tool = DirectoryReadTool(directory="./data/papers")


agent_archivist = Agent(
    role="Team Archivist",
    goal="Locate the requested document in the folder—even with partial, approximate, or misspelled names—and extract its raw content.",
    backstory="""You are an extremely meticulous and agile archivist. You understand that users 
    might not remember the exact name of a file. 
    
    CRITICAL PROTOCOL:
    1. When given a filename or a search query, you MUST ALWAYS start by listing all files in the directory using the DirectoryReadTool.
    2. Analyze the list of filenames and find the most logical match. Handle typos, partial matches, variations, or files missing extensions (e.g., if the user asks for 'sales report' and the folder has 'sales_report_2026_v2.txt', you should select it).
    3. Once you identify the best match, use the FileReadTool to read its content.
    4. If no clear match is found, report the available filenames to the team.""",
    tools=[directory_tool, read_pdf_content],
    llm=llm,
    max_iter=2,
    verbose=True
)
save_bib_tool = SaveBibTeXTool()

agent_extractor = Agent(
    role="Academic Metadata Publisher",
    goal="Extract key bibliographic entities, fill the structured database, and save it as a .bib file.",
    backstory="""You are an expert research librarian. You take raw academic text, 
    extract its metadata, and use specialized saving tools to record them. 
    You are responsible for choosing an appropriate file name (like 'FirstAuthorYear.bib') 
    and ensuring all citation details are perfectly recorded in the database.""",
    tools=[save_bib_tool],
    llm=llm,
    max_iter=2,
    verbose=True
)


## 7. Tarefas
- **task_busca**: busca o arquivo no diretório e retorna seu conteúdo bruto.
- **task_extraction**: extrai metadados do texto e retorna JSON conforme `BibliographyEntry`.

In [10]:
task_busca = Task(
    description="""Search the documents folder for a file that matches or is highly similar to the query '{filename}'.
    First, list all files in the directory. Then, evaluate and identify the single best matching file (even if the query is partial, misspelled, or lacks the extension). 
    Finally, read and return the entire content of that matched file.""",
    expected_output="The raw text content of the best-matched file found in the directory.",
    agent=agent_archivist
)

task_extraction = Task(
    description="""Analyze the raw text provided by the Archivist.
    Identify and extract all key academic metadata entities. 
    Execute the 'Save BibTeX Tool' by passing all the extracted fields and your chosen filename 
    to successfully save the reference in the directory.""",
    expected_output="The confirmation message from the Save BibTeX Tool showing where the file was saved.",
    agent=agent_extractor
)

## 8. Montagem da Crew
Cria o `Crew` com os dois agentes e duas tarefas em execução **sequencial** (busca → extração).

In [11]:
crew = Crew(
    agents=[agent_archivist, agent_extractor],
    tasks=[task_busca, task_extraction],
    process=Process.sequential, 
    verbose=True
)


## 9. Execução
Dispara o pipeline completo: o Archivist encontra e lê o PDF, depois o Extractor gera a entrada bibliográfica estruturada.

In [ ]:

resultado = await crew.kickoff_async(inputs={"filename": "langgraph paper"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 00853d0d-9a85-45e5-b513-172b20457037                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search the documents folder for a file that matches or is highly similar to the query 'langgraph         │
│  paper'.                                                                                                        │
│      First, list all files in the directory. Then, evaluate and identify the single best matching file (even    │
│  if the query is partial, misspelled, or lacks the extension).                                                  │
│      Finally, read and return the entire content of that matched file.                                          │
│  ID: 8162a964-72d6-4e7b-b6c9-9692067592d0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Team Archivist                                                                                          │
│                                                                                                                 │
│  Task: Search the documents folder for a file that matches or is highly similar to the query 'langgraph         │
│  paper'.                                                                                                        │
│      First, list all files in the directory. Then, evaluate and identify the single best matching file (even    │
│  if the query is partial, misspelled, or lacks the extension).                                                  │
│      Finally, read and return the entire content of that matched file.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: list_files_in_directory                                                                                  │
│  Args: {}                                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: list_files_in_directory                                                                                  │
│  Output: File paths:                                                                                            │
│  -/home/clara/Documentos/AgenteExtrator/data/papers/hierarchicalMultiAgent.pdf                                  │
│  - /home/clara/Documentos/AgenteExtrator/data/papers/agentAILanggraph.pdf                                       │
│  - /home/clara/Documentos/AgenteExtrator/data/papers/Multi-agentembodiedAI:.pdf                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Team Archivist                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  LangGraph: A Graph-Based Language Model for Generative Conversational AI                                       │
│                                                                                                                 │
│   Abstract                                                                                                      │
│                                                                                                                 │
│   We propose LangGraph, a novel graph-based language model for generative conversational AI. LangGraph          │
│  combines the strengths of graph neural networks and language models to generate coherent and                   │
│  context-dependent responses. Our model consists of two main components: a graph encoder and a language         │
│  decoder. The graph encoder captures the structural relationships between entities in the conversation          │
│  history, while the language decoder generates responses based on the encoded graph representation. We          │
│  evaluate LangGraph on several conversational AI benchmarks and demonstrate its effectiveness in generating     │
│  coherent and engaging responses.                                                                               │
│                                                                                                                 │
│   Introduction                                                                                                  │
│                                                                                                                 │
│   Conversational AI has become increasingly popular in recent years, with applications in customer service,     │
│  language translation, and social companionship. However, existing conversational AI models often struggle to   │
│  generate coherent and context-dependent responses, particularly in multi-turn conversations. To address this   │
│  challenge, we propose LangGraph, a graph-based language model that combines the strengths of graph neural      │
│  networks and language models.                                                                                  │
│                                                                                                                 │
│   Graph Encoder                                                                                                 │
│                                                                                                                 │
│   The graph encoder is responsible for capturing the structural relationships between entities in the           │
│  conversation history. We represent the conversation history as a graph, where each node corresponds to an      │
│  entity (e.g., a person, a location, or an object), and each edge represents a relationship between entities    │
│  (e.g., "person A is talking to person B"). We use a graph neural network to encode the graph representation    │
│  into a continuous vector space.                                                                                │
│                                                                                                                 │
│   Language Decoder                                                                                              │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Search the documents folder for a file that matches or is highly similar to the query 'langgraph         │
│  paper'.                                                                                                        │
│      First, list all files in the directory. Then, evaluate and identify the single best matching file (even    │
│  if the query is partial, misspelled, or lacks the extension).                                                  │
│      Finally, read and return the entire content of that matched file.                                          │
│  Agent: Team Archivist                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the raw text provided by the Archivist.                                                          │
│      Identify and extract all key academic metadata entities.                                                   │
│      Execute the 'Save BibTeX Tool' by passing all the extracted fields and your chosen filename                │
│      to successfully save the reference in the directory.                                                       │
│  ID: f848fd6b-7a43-46c9-81fc-c12cbf48132e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Academic Metadata Publisher                                                                             │
│                                                                                                                 │
│  Task: Analyze the raw text provided by the Archivist.                                                          │
│      Identify and extract all key academic metadata entities.                                                   │
│      Execute the 'Save BibTeX Tool' by passing all the extracted fields and your chosen filename                │
│      to successfully save the reference in the directory.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯